# Phase 3: Functional Enrichment

This notebook separates the enrichment analysis from the earlier combined notebook and starts from the Phase 2 DEG outputs saved under `results/`.

Current scope in this repo:
- GO Biological Process enrichment for each breast cancer subtype
- subtype-specific dot plots for activated and suppressed GO terms

Deferred for now:
- KEGG enrichment
- KEGG pathway map overlays


## Inputs And Outputs

Expected inputs:
- `results/DEGs_Basal_vs_Normal.csv`
- `results/DEGs_Her2_vs_Normal.csv`
- `results/DEGs_LumA_vs_Normal.csv`
- `results/DEGs_LumB_vs_Normal.csv`

Outputs written by this notebook:
- `results/phase3/go_bp_Basal_activated.csv`
- `results/phase3/go_bp_Basal_suppressed.csv`
- `results/phase3/go_bp_Her2_activated.csv`
- `results/phase3/go_bp_Her2_suppressed.csv`
- `results/phase3/go_bp_LumA_activated.csv`
- `results/phase3/go_bp_LumA_suppressed.csv`
- `results/phase3/go_bp_LumB_activated.csv`
- `results/phase3/go_bp_LumB_suppressed.csv`
- `results/phase3/go_bp_all_subtypes.csv`
- `figures/phase3_go_bp_Basal_dot.png`
- `figures/phase3_go_bp_Her2_dot.png`
- `figures/phase3_go_bp_LumA_dot.png`
- `figures/phase3_go_bp_LumB_dot.png`

Note: running Enrichr through `gseapy` requires internet access.


In [1]:
from pathlib import Path
import os
import warnings

mpl_cache = Path.cwd() / ".mplconfig"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

try:
    import gseapy as gp
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gseapy"])
    import gseapy as gp

FIG_DIR = Path("figures")
RES_DIR = Path("results/phase3")
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

TUMOR_SUBTYPES = ["Basal", "Her2", "LumA", "LumB"]
GO_LIBRARY = "GO_Biological_Process_2023"

required_deg_files = {
    subtype: Path(f"results/DEGs_{subtype}_vs_Normal.csv")
    for subtype in TUMOR_SUBTYPES
}
missing = [str(path) for path in required_deg_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing Phase 2 DEG outputs: " + ", ".join(missing) +
        ". Run phase2_differential_expression.ipynb first."
    )


## Load Significant DEG Tables

We use the significant subtype-vs-normal DEG lists from Phase 2 as the input gene sets for enrichment.


In [2]:
sig_deg = {
    subtype: pd.read_csv(path)
    for subtype, path in required_deg_files.items()
}

deg_gene_sets = {
    subtype: sorted(sig_deg[subtype]["gene_name"].dropna().astype(str).unique().tolist())
    for subtype in TUMOR_SUBTYPES
}

pd.DataFrame(
    {
        "Subtype": TUMOR_SUBTYPES,
        "Genes in DEG set": [len(deg_gene_sets[subtype]) for subtype in TUMOR_SUBTYPES],
    }
).set_index("Subtype")


,Genes in DEG set
Subtype,
Basal,5790
Her2,5714
LumA,4064
LumB,5982


## GO Biological Process Enrichment

Each subtype is split into activated and suppressed genes using the sign of the log2 fold change, then enriched against `GO_Biological_Process_2023` with Enrichr via `gseapy`.


In [3]:
go_results = []

for subtype in TUMOR_SUBTYPES:
    df_sig = sig_deg[subtype].copy()

    up_genes = df_sig.loc[df_sig["log2FoldChange"] > 0, "gene_name"].dropna().astype(str).unique().tolist()
    down_genes = df_sig.loc[df_sig["log2FoldChange"] < 0, "gene_name"].dropna().astype(str).unique().tolist()

    for direction, genes in [("Activated", up_genes), ("Suppressed", down_genes)]:
        if len(genes) < 10:
            print(f"Skipping {subtype} {direction}: too few genes for GO BP ({len(genes)}).")
            continue

        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=[GO_LIBRARY],
            organism="human",
            outdir=None,
            cutoff=0.5,
        )

        df = enr.results.copy()
        df["Subtype"] = subtype
        df["Direction"] = direction

        if "Overlap" not in df.columns and "Genes" in df.columns:
            df["Overlap"] = df["Genes"].astype(str).apply(
                lambda value: str(len(value.split(";")) if value else 0) + "/0"
            )

        if df["Adjusted P-value"].isna().all() and "Old adjusted P-value" in df.columns:
            df["Adjusted P-value"] = df["Old adjusted P-value"]
            if "Old P-value" in df.columns:
                df["P-value"] = df["Old P-value"]

        df = df.sort_values("Adjusted P-value", ascending=True)
        df.to_csv(RES_DIR / f"go_bp_{subtype}_{direction.lower()}.csv", index=False)
        go_results.append(df)

go_all = pd.concat(go_results, ignore_index=True) if go_results else pd.DataFrame()
if not go_all.empty:
    go_all.to_csv(RES_DIR / "go_bp_all_subtypes.csv", index=False)
    print(f"Saved GO BP enrichment: {go_all.shape[0]:,} rows total")
else:
    print("No GO BP enrichment results generated.")


Saved GO BP enrichment: 26,136 rows total


## Plot The Enriched GO Terms

For each subtype, the notebook creates a paired dot plot showing the strongest activated and suppressed GO Biological Process terms.


In [4]:
if go_all.empty:
    print("No GO BP results to plot.")
else:
    for subtype in TUMOR_SUBTYPES:
        sub_df = go_all[go_all["Subtype"] == subtype].copy()
        if sub_df.empty:
            print(f"Skipping {subtype}: no GO BP terms available.")
            continue

        sub_df["Count"] = sub_df["Overlap"].astype(str).str.split("/").str[0].astype(float)
        sub_df["neg_log10_padj"] = -np.log10(sub_df["Adjusted P-value"].clip(lower=1e-300))

        activated = (
            sub_df[sub_df["Direction"] == "Activated"]
            .sort_values("Adjusted P-value", ascending=True)
            .head(10)
            .copy()
        )
        suppressed = (
            sub_df[sub_df["Direction"] == "Suppressed"]
            .sort_values("Adjusted P-value", ascending=True)
            .head(10)
            .copy()
        )

        if activated.empty and suppressed.empty:
            print(f"Skipping {subtype}: no activated/suppressed GO terms available.")
            continue

        fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=False)
        panels = [(axes[0], activated, "Activated"), (axes[1], suppressed, "Suppressed")]
        scatter = None

        for ax, panel_df, label in panels:
            if panel_df.empty:
                ax.set_title(label)
                ax.text(0.5, 0.5, "No terms", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue

            panel_df = panel_df.sort_values("neg_log10_padj", ascending=True)
            y = np.arange(len(panel_df))
            scatter = ax.scatter(
                panel_df["neg_log10_padj"],
                y,
                s=panel_df["Count"] * 8,
                c=panel_df["neg_log10_padj"],
                cmap="plasma",
                alpha=0.9,
                edgecolor="black",
                linewidth=0.4,
            )
            ax.set_yticks(y)
            ax.set_yticklabels(panel_df["Term"], fontsize=8)
            ax.set_xlabel("-log10(Adjusted P-value)")
            ax.set_title(label)
            ax.grid(alpha=0.25)

        fig.suptitle(f"{subtype}: GO Biological Process Enrichment", y=1.02, fontsize=14)
        fig.tight_layout()

        if scatter is not None:
            cbar = fig.colorbar(scatter, ax=axes, shrink=0.8, pad=0.02)
            cbar.set_label("Significance: -log10(adj p)")

        out_path = FIG_DIR / f"phase3_go_bp_{subtype}_dot.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved GO BP dot plot: {out_path}")


Saved GO BP dot plot: figures\phase3_go_bp_Basal_dot.png
Saved GO BP dot plot: figures\phase3_go_bp_Her2_dot.png
Saved GO BP dot plot: figures\phase3_go_bp_LumA_dot.png
Saved GO BP dot plot: figures\phase3_go_bp_LumB_dot.png


In [5]:
KEGG_LIBRARY = "KEGG_2021_Human"

kegg_results = []

for subtype in TUMOR_SUBTYPES:
    df_sig = sig_deg[subtype].copy()

    up_genes = df_sig.loc[df_sig["log2FoldChange"] > 0, "gene_name"].dropna().astype(str).unique().tolist()
    down_genes = df_sig.loc[df_sig["log2FoldChange"] < 0, "gene_name"].dropna().astype(str).unique().tolist()

    for direction, genes in [("Activated", up_genes), ("Suppressed", down_genes)]:
        if len(genes) < 10:
            print(f"Skipping {subtype} {direction}: too few genes for KEGG ({len(genes)}).")
            continue

        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=[KEGG_LIBRARY],
            organism="human",
            outdir=None,
            cutoff=0.5,
        )

        df = enr.results.copy()
        df["Subtype"] = subtype
        df["Direction"] = direction

        df = df.sort_values("Adjusted P-value", ascending=True)
        df.to_csv(RES_DIR / f"kegg_{subtype}_{direction.lower()}.csv", index=False)

        kegg_results.append(df)

kegg_all = pd.concat(kegg_results, ignore_index=True)
kegg_all.to_csv(RES_DIR / "kegg_all_subtypes.csv", index=False)

print(f"Saved KEGG enrichment: {kegg_all.shape[0]} rows")

Saved KEGG enrichment: 2251 rows


In [8]:
kegg_tables = {}

for subtype in TUMOR_SUBTYPES:
    for direction in ["Activated", "Suppressed"]:
        key = f"{subtype}_{direction}"
        
        kegg_tables[key] = kegg_all[
            (kegg_all["Subtype"] == subtype) &
            (kegg_all["Direction"] == direction)
        ].sort_values("Adjusted P-value")

kegg_tables["LumA_Activated"].head()

,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Subtype,Direction
1170,KEGG_2021_Human,Systemic lupus erythematosus,28/135,0.000004,0.000968,0,0,2.953109,36.567564,H2AC19;H2AC17;H2AC18;C4B;C4A;H2AC13;H3C15;H3C1...,LumA,Activated
1171,KEGG_2021_Human,Alcoholism,29/186,0.000620,0.071607,0,0,2.080083,15.363140,H2AC19;H2AC17;H2AC18;GNGT1;H2AC13;H3C15;H3C13;...,LumA,Activated
1172,KEGG_2021_Human,Neutrophil extracellular trap formation,27/189,0.003406,0.262292,0,0,1.874021,10.648378,H2AC19;H2AC17;H2AC18;H2AC13;H3C15;H3C13;H3C14;...,LumA,Activated
1173,KEGG_2021_Human,Caffeine metabolism,3/6,0.009189,0.530659,0,0,11.176614,52.415625,CYP2A7;CYP2A6;NAT1,LumA,Activated
1174,KEGG_2021_Human,Maturity onset diabetes of the young,5/26,0.057642,0.999996,0,0,2.661731,7.595278,NKX6-1;MAFA;FOXA3;NKX2-2;NEUROG3,LumA,Activated


In [9]:

#filter pathways that are significant 
sig_kegg = kegg_all[kegg_all["Adjusted P-value"] < 0.05]
lumA = sig_kegg[sig_kegg["Subtype"] == "LumA"]
lumA = lumA.sort_values("Adjusted P-value")
lumA.head(5)[["Term", "Adjusted P-value", "Overlap"]]


,Term,Adjusted P-value,Overlap
1170,Systemic lupus erythematosus,0.000968,28/135
1401,PPAR signaling pathway,0.000990,24/74
1402,Retinol metabolism,0.046320,19/68


In [15]:
def export_kegg_colors(df_sig, subtype):
    file_path = RES_DIR / f"kegg_colors_{subtype}.txt"

    with open(file_path, "w") as f:
        for _, row in df_sig.iterrows():
            gene = row["gene_name"]
            logfc = row["log2FoldChange"]

            if pd.isna(gene):
                continue

            if logfc > 0:
                f.write(f"{gene}\tred\n")
            else:
                f.write(f"{gene}\tblue\n")

    print(f"Saved: {file_path}")
    return file_path
lumA_file = export_kegg_colors(sig_deg["LumA"], "LumA")
Basal_file = export_kegg_colors(sig_deg["Basal"], "Basal")
Her2_file = export_kegg_colors(sig_deg["Her2"], "Her2") 
lumB_file = export_kegg_colors(sig_deg["LumB"], "LumB")

Saved: results\phase3\kegg_colors_LumA.txt
Saved: results\phase3\kegg_colors_Basal.txt
Saved: results\phase3\kegg_colors_Her2.txt
Saved: results\phase3\kegg_colors_LumB.txt


In [12]:
basal = sig_kegg[sig_kegg["Subtype"] == "Basal"]
basal = basal.sort_values("Adjusted P-value")
basal.head(10)[["Term", "Adjusted P-value", "Overlap", "Direction"]]

,Term,Adjusted P-value,Overlap,Direction
293,Tyrosine metabolism,0.000032,18/36,Suppressed
0,Systemic lupus erythematosus,0.000078,45/135,Activated
1,Cell cycle,0.000078,42/124,Activated
294,Metabolism of xenobiotics by cytochrome P450,0.000083,27/76,Suppressed
295,Retinol metabolism,0.000963,23/68,Suppressed
296,Drug metabolism,0.007584,29/108,Suppressed
297,Regulation of lipolysis in adipocytes,0.008432,18/55,Suppressed
298,ABC transporters,0.019699,15/45,Suppressed
2,Alcoholism,0.028886,48/186,Activated
299,Phenylalanine metabolism,0.029119,8/17,Suppressed


In [13]:
LumB = sig_kegg[sig_kegg["Subtype"] == "LumB"]
LumB = LumB.sort_values("Adjusted P-value")
LumB.head(10)[["Term", "Adjusted P-value", "Overlap", "Direction"]]

,Term,Adjusted P-value,Overlap,Direction
1686,Systemic lupus erythematosus,5.450925e-07,42/135,Activated
1687,Alcoholism,4.186834e-05,47/186,Activated
1688,Neutrophil extracellular trap formation,6.754368e-04,44/189,Activated
1689,Cell cycle,6.460733e-03,30/124,Activated


In [14]:
Her2 = sig_kegg[sig_kegg["Subtype"] == "Her2"]
Her2 = Her2.sort_values("Adjusted P-value")
Her2.head(10)[["Term", "Adjusted P-value", "Overlap", "Direction"]]

,Term,Adjusted P-value,Overlap,Direction
581,Systemic lupus erythematosus,2.180083e-12,53/135,Activated
582,Alcoholism,1.585212e-09,59/186,Activated
583,Neutrophil extracellular trap formation,1.510406e-06,53/189,Activated
876,Regulation of lipolysis in adipocytes,4.043655e-03,22/55,Suppressed
